In [1]:
import pandas as pd
import numpy as np
import os
from pathlib import Path

print("FlightFore - Week 2")
print("Preprocessing environment ready!")

FlightFore - Week 2
Preprocessing environment ready!


In [3]:
from pathlib import Path

dataset_path = Path(r"E:\Second year KLH Softwares\ML\Flightfore Project\data\raw\flight_ontime_reporting_with_weather")

csv_files = list(dataset_path.glob("*.csv"))

print("Number of CSV files:", len(csv_files))

print("\nFirst 20 files:")
for file in csv_files[:20]:
    print(file.name)

Number of CSV files: 30

First 20 files:
ATL.csv
AUS.csv
BNA.csv
BOS.csv
BWI.csv
CLT.csv
DCA.csv
DEN.csv
DFW.csv
DTW.csv
EWR.csv
FLL.csv
IAD.csv
IAH.csv
JFK.csv
LAS.csv
LAX.csv
LGA.csv
MCO.csv
MDW.csv


In [4]:
# Check the columns of every airport file

column_sets = {}

for file in csv_files:
    df_temp = pd.read_csv(file, nrows=5)
    column_sets[file.name] = list(df_temp.columns)

# Check whether all files have identical columns
first_columns = list(column_sets.values())[0]

all_same = all(columns == first_columns for columns in column_sets.values())

print("Do all files have the same columns?", all_same)

if not all_same:
    print("\nFiles with different columns:")
    for filename, columns in column_sets.items():
        if columns != first_columns:
            print(filename)

Do all files have the same columns? True


In [5]:
# Combine all airport CSV files

dataframes = []

for file in csv_files:
    print(f"Loading: {file.name}")
    
    df_temp = pd.read_csv(file)
    dataframes.append(df_temp)

# Combine all airport datasets
flightfore = pd.concat(dataframes, ignore_index=True)

print("\nCombined dataset created!")
print("Shape:", flightfore.shape)

Loading: ATL.csv
Loading: AUS.csv
Loading: BNA.csv
Loading: BOS.csv
Loading: BWI.csv
Loading: CLT.csv
Loading: DCA.csv
Loading: DEN.csv
Loading: DFW.csv
Loading: DTW.csv
Loading: EWR.csv
Loading: FLL.csv
Loading: IAD.csv
Loading: IAH.csv
Loading: JFK.csv
Loading: LAS.csv


C:\Users\giris\AppData\Local\Temp\ipykernel_50156\916694707.py:8: DtypeWarning: Columns (0: CancellationReason) have mixed types. Specify dtype option on import or set low_memory=False.
  df_temp = pd.read_csv(file)


Loading: LAX.csv
Loading: LGA.csv
Loading: MCO.csv
Loading: MDW.csv
Loading: MIA.csv
Loading: MSP.csv
Loading: ORD.csv
Loading: PHL.csv
Loading: PHX.csv
Loading: SAN.csv
Loading: SEA.csv
Loading: SFO.csv
Loading: SLC.csv
Loading: TPA.csv

Combined dataset created!
Shape: (14494043, 22)


In [6]:
import gc

del dataframes
gc.collect()

print("Individual airport DataFrames released from memory.")
print("FlightFore shape:", flightfore.shape)

Individual airport DataFrames released from memory.
FlightFore shape: (14494043, 22)


## Inspection

In [7]:
print("Shape:", flightfore.shape)

print("\nData types:")
print(flightfore.dtypes)

print("\nMemory usage:")
print(f"{flightfore.memory_usage(deep=True).sum() / (1024**3):.2f} GB")

Shape: (14494043, 22)

Data types:
Time                          str
Origin                        str
Dest                          str
Carrier                       str
Cancelled                    bool
CancellationReason            str
Delayed                      bool
DepDelayMinutes           float64
CarrierDelay              float64
WeatherDelay              float64
NASDelay                  float64
SecurityDelay             float64
LateAircraftDelay         float64
Temperature               float64
Feels_Like_Temperature    float64
Altimeter_Pressure        float64
Sea_Level_Pressure        float64
Visibility                float64
Wind_Speed                float64
Wind_Gust                 float64
Precipitation             float64
Ice_Accretion_3hr         float64
dtype: object

Memory usage:
5.77 GB


In [8]:
print("\nMissing values:")
print(flightfore.isnull().sum())


Missing values:
Time                             0
Origin                           0
Dest                             0
Carrier                          0
Cancelled                        0
CancellationReason        14162151
Delayed                          0
DepDelayMinutes             321246
CarrierDelay              11336160
WeatherDelay              11336160
NASDelay                  11336160
SecurityDelay             11336160
LateAircraftDelay         11336160
Temperature                      0
Feels_Like_Temperature           0
Altimeter_Pressure               0
Sea_Level_Pressure               0
Visibility                       0
Wind_Speed                       0
Wind_Gust                        0
Precipitation                    0
Ice_Accretion_3hr                0
dtype: int64


In [9]:
print("\nDuplicate rows:", flightfore.duplicated().sum())


Duplicate rows: 54744


In [10]:
print("Cancellation counts:")
print(flightfore["Cancelled"].value_counts())

print("\nDelayed counts:")
print(flightfore["Delayed"].value_counts())

print("\nDepDelayMinutes missing by cancellation status:")
print(
    flightfore.groupby("Cancelled")["DepDelayMinutes"]
    .apply(lambda x: x.isna().sum())
)

print("\nDepDelayMinutes available by cancellation status:")
print(
    flightfore.groupby("Cancelled")["DepDelayMinutes"]
    .apply(lambda x: x.notna().sum())
)

Cancellation counts:
Cancelled
False    14162151
True       331892
Name: count, dtype: int64

Delayed counts:
Delayed
False    8834900
True     5659143
Name: count, dtype: int64

DepDelayMinutes missing by cancellation status:
Cancelled
False         0
True     321246
Name: DepDelayMinutes, dtype: int64

DepDelayMinutes available by cancellation status:
Cancelled
False    14162151
True        10646
Name: DepDelayMinutes, dtype: int64


In [11]:
duplicate_rows = flightfore[flightfore.duplicated(keep=False)]

print("Number of rows involved in duplicates:", len(duplicate_rows))

print("\nFirst 10 duplicate rows:")
display(duplicate_rows.head(10))

Number of rows involved in duplicates: 108484

First 10 duplicate rows:


,Time,Origin,Dest,Carrier,Cancelled,CancellationReason,Delayed,DepDelayMinutes,CarrierDelay,WeatherDelay,...,LateAircraftDelay,Temperature,Feels_Like_Temperature,Altimeter_Pressure,Sea_Level_Pressure,Visibility,Wind_Speed,Wind_Gust,Precipitation,Ice_Accretion_3hr
24726,2021-01-10 07:00:00,ATL,ORD,Republic Airline,False,NaN,False,0.0,NaN,NaN,...,NaN,28.0,21.64,1024.72,1025.6,16093.4,5.75,24.74,0.0,0.0
24732,2021-01-17 07:00:00,ATL,ORD,Republic Airline,False,NaN,False,0.0,NaN,NaN,...,NaN,33.0,28.70,1014.56,1015.3,16093.4,4.60,24.74,0.0,0.0
24746,2021-01-03 18:00:00,ATL,ORD,Republic Airline,False,NaN,False,0.0,NaN,NaN,...,NaN,45.0,41.19,1017.95,1018.4,16093.4,6.90,24.74,0.0,0.0
24747,2021-01-04 18:00:00,ATL,ORD,Republic Airline,False,NaN,False,0.0,NaN,NaN,...,NaN,54.0,54.00,1016.59,1017.1,16093.4,3.45,24.74,0.0,0.0
24748,2021-01-06 16:00:00,ATL,ORD,Republic Airline,False,NaN,False,0.0,NaN,NaN,...,NaN,55.0,55.00,1019.98,1020.5,16093.4,5.75,24.74,0.0,0.0
24756,2021-01-12 16:00:00,ATL,ORD,Republic Airline,False,NaN,False,0.0,NaN,NaN,...,NaN,43.0,37.66,1022.69,1023.3,16093.4,9.21,24.74,0.0,0.0
24757,2021-01-12 16:00:00,ATL,ORD,Republic Airline,False,NaN,False,0.0,NaN,NaN,...,NaN,43.0,37.19,1022.69,1018.0,16093.4,10.36,24.74,0.0,0.0
24763,2021-01-19 16:00:00,ATL,ORD,Republic Airline,False,NaN,False,0.0,NaN,NaN,...,NaN,60.0,60.10,1025.74,1025.9,16093.4,8.06,24.74,0.0,0.0
24764,2021-01-20 16:00:00,ATL,ORD,Republic Airline,False,NaN,False,0.0,NaN,NaN,...,NaN,55.0,55.00,1022.69,1022.7,16093.4,9.21,24.74,0.0,0.0
24778,2021-01-27 16:00:00,ATL,ORD,Republic Airline,False,NaN,False,0.0,NaN,NaN,...,NaN,59.0,59.00,1014.22,1014.1,16093.4,5.75,24.74,0.0,0.0


In [12]:
duplicate_only = flightfore[flightfore.duplicated(keep=False)]

print("Rows involved in duplicate groups:", len(duplicate_only))

print("\nNumber of duplicate rows that would be removed:",
      flightfore.duplicated().sum())

print("\nMaximum number of identical copies of one record:")

# Check how many times each duplicate row occurs
duplicate_counts = (
    duplicate_only
    .groupby(list(flightfore.columns), dropna=False)
    .size()
)

print(duplicate_counts.max())

Rows involved in duplicate groups: 108484

Number of duplicate rows that would be removed: 54744

Maximum number of identical copies of one record:
6


## Remove Exact Duplicates

In [13]:
print("Shape before removing duplicates:", flightfore.shape)

duplicates_before = flightfore.duplicated().sum()
print("Duplicate rows to remove:", duplicates_before)

flightfore.drop_duplicates(inplace=True)

print("\nShape after removing duplicates:", flightfore.shape)
print("Duplicate rows remaining:", flightfore.duplicated().sum())

Shape before removing duplicates: (14494043, 22)
Duplicate rows to remove: 54744

Shape after removing duplicates: (14439299, 22)
Duplicate rows remaining: 0


In [14]:
print("Cancellation distribution:")
print(flightfore["Cancelled"].value_counts())

print("\nCancellation percentage:")
print(flightfore["Cancelled"].value_counts(normalize=True) * 100)

Cancellation distribution:
Cancelled
False    14108696
True       330603
Name: count, dtype: int64

Cancellation percentage:
Cancelled
False    97.710394
True      2.289606
Name: proportion, dtype: float64


In [15]:
# ============================================
# STEP 3: CREATE CLASSIFICATION & REGRESSION DATA
# ============================================

# Classification dataset
# Goal: predict whether a flight will be cancelled
classification_data = flightfore.copy()

print("CLASSIFICATION DATASET")
print("Shape:", classification_data.shape)
print("\nTarget distribution:")
print(classification_data["Cancelled"].value_counts())
print("\nTarget percentage:")
print(classification_data["Cancelled"].value_counts(normalize=True) * 100)


# Regression dataset
# Goal: predict departure delay in minutes
# Only non-cancelled flights have a meaningful departure delay

regression_data = flightfore[
    (flightfore["Cancelled"] == False) &
    (flightfore["DepDelayMinutes"].notna())
].copy()

print("\n" + "="*50)
print("REGRESSION DATASET")
print("Shape:", regression_data.shape)

print("\nMissing DepDelayMinutes:")
print(regression_data["DepDelayMinutes"].isna().sum())

print("\nDepDelayMinutes statistics:")
print(regression_data["DepDelayMinutes"].describe())

CLASSIFICATION DATASET
Shape: (14439299, 22)

Target distribution:
Cancelled
False    14108696
True       330603
Name: count, dtype: int64

Target percentage:
Cancelled
False    97.710394
True      2.289606
Name: proportion, dtype: float64

REGRESSION DATASET
Shape: (14108696, 22)

Missing DepDelayMinutes:
0

DepDelayMinutes statistics:
count    1.410870e+07
mean     1.621702e+01
std      5.162100e+01
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      1.100000e+01
max      3.695000e+03
Name: DepDelayMinutes, dtype: float64


In [16]:
# ============================================
# STEP 4: TIME FEATURE ENGINEERING
# ============================================

# Convert Time to datetime
flightfore["Time"] = pd.to_datetime(flightfore["Time"])

# Create time-based features
flightfore["Hour"] = flightfore["Time"].dt.hour
flightfore["DayOfWeek"] = flightfore["Time"].dt.dayofweek
flightfore["Month"] = flightfore["Time"].dt.month
flightfore["DayOfMonth"] = flightfore["Time"].dt.day

# Weekend indicator
flightfore["IsWeekend"] = (
    flightfore["DayOfWeek"] >= 5
).astype(int)

print("Time feature engineering completed.")

print("\nNew time features:")
print(
    flightfore[
        [
            "Time",
            "Hour",
            "DayOfWeek",
            "Month",
            "DayOfMonth",
            "IsWeekend"
        ]
    ].head()
)

print("\nTime feature ranges:")
print("Hour:", flightfore["Hour"].min(), "to", flightfore["Hour"].max())
print("DayOfWeek:", flightfore["DayOfWeek"].min(), "to", flightfore["DayOfWeek"].max())
print("Month:", flightfore["Month"].min(), "to", flightfore["Month"].max())

Time feature engineering completed.

New time features:
                 Time  Hour  DayOfWeek  Month  DayOfMonth  IsWeekend
0 2021-01-01 14:00:00    14          4      1           1          0
1 2021-01-01 14:00:00    14          4      1           1          0
2 2021-01-02 14:00:00    14          5      1           2          1
3 2021-01-03 14:00:00    14          6      1           3          1
4 2021-01-04 14:00:00    14          0      1           4          0

Time feature ranges:
Hour: 0 to 23
DayOfWeek: 0 to 6
Month: 1 to 12


In [17]:
# ============================================
# STEP 5: DEFINE LEAKAGE-SAFE FEATURES
# ============================================

feature_columns = [
    # Flight features
    "Origin",
    "Dest",
    "Carrier",

    # Time features
    "Hour",
    "DayOfWeek",
    "Month",
    "DayOfMonth",
    "IsWeekend",

    # Weather features
    "Temperature",
    "Feels_Like_Temperature",
    "Altimeter_Pressure",
    "Sea_Level_Pressure",
    "Visibility",
    "Wind_Speed",
    "Wind_Gust",
    "Precipitation",
    "Ice_Accretion_3hr"
]

print("Number of prediction features:", len(feature_columns))

print("\nPrediction features:")
for i, feature in enumerate(feature_columns, start=1):
    print(f"{i:2}. {feature}")

Number of prediction features: 17

Prediction features:
 1. Origin
 2. Dest
 3. Carrier
 4. Hour
 5. DayOfWeek
 6. Month
 7. DayOfMonth
 8. IsWeekend
 9. Temperature
10. Feels_Like_Temperature
11. Altimeter_Pressure
12. Sea_Level_Pressure
13. Visibility
14. Wind_Speed
15. Wind_Gust
16. Precipitation
17. Ice_Accretion_3hr


In [18]:
# Verify that all selected features exist
missing_features = [
    feature for feature in feature_columns
    if feature not in flightfore.columns
]

if len(missing_features) == 0:
    print("✓ All prediction features exist in the dataset.")
else:
    print("Missing features:", missing_features)

✓ All prediction features exist in the dataset.


In [19]:
import gc

# Release unnecessary full-dataframe copies
if "classification_data" in globals():
    del classification_data

if "regression_data" in globals():
    del regression_data

gc.collect()

print("Unnecessary dataframe copies released.")
print("Master dataset shape:", flightfore.shape)

Unnecessary dataframe copies released.
Master dataset shape: (14439299, 27)


In [20]:
feature_missing = flightfore[feature_columns].isnull().sum()

print("Missing values in prediction features:")
print(feature_missing)

print("\nTotal missing feature values:",
      feature_missing.sum())

Missing values in prediction features:
Origin                    0
Dest                      0
Carrier                   0
Hour                      0
DayOfWeek                 0
Month                     0
DayOfMonth                0
IsWeekend                 0
Temperature               0
Feels_Like_Temperature    0
Altimeter_Pressure        0
Sea_Level_Pressure        0
Visibility                0
Wind_Speed                0
Wind_Gust                 0
Precipitation             0
Ice_Accretion_3hr         0
dtype: int64

Total missing feature values: 0


In [21]:
# ============================================
# CHECK TIME RANGE
# ============================================

print("Earliest flight time:")
print(flightfore["Time"].min())

print("\nLatest flight time:")
print(flightfore["Time"].max())

print("\nNumber of unique dates:")
print(flightfore["Time"].dt.date.nunique())

Earliest flight time:
2021-01-01 01:00:00

Latest flight time:
2023-12-31 00:00:00

Number of unique dates:
1095


In [22]:
# Check number of records by year

print("\nRecords by year:")
print(flightfore["Time"].dt.year.value_counts().sort_index())


Records by year:
Time
2021    4230366
2022    5036769
2023    5172164
Name: count, dtype: int64


In [23]:
# ============================================
# TEMPORAL TRAIN / VALIDATION / TEST SPLIT
# ============================================

train_mask = flightfore["Time"].dt.year == 2021
val_mask = flightfore["Time"].dt.year == 2022
test_mask = flightfore["Time"].dt.year == 2023

print("TRAIN:", train_mask.sum())
print("VALIDATION:", val_mask.sum())
print("TEST:", test_mask.sum())

print("\nTotal:")
print(train_mask.sum() + val_mask.sum() + test_mask.sum())

print("\nExpected total:")
print(len(flightfore))

TRAIN: 4230366
VALIDATION: 5036769
TEST: 5172164

Total:
14439299

Expected total:
14439299


In [24]:
# ============================================
#  DEFINE FEATURE GROUPS
# ============================================

categorical_features = [
    "Origin",
    "Dest",
    "Carrier"
]

numerical_features = [
    "Hour",
    "DayOfWeek",
    "Month",
    "DayOfMonth",
    "IsWeekend",
    "Temperature",
    "Feels_Like_Temperature",
    "Altimeter_Pressure",
    "Sea_Level_Pressure",
    "Visibility",
    "Wind_Speed",
    "Wind_Gust",
    "Precipitation",
    "Ice_Accretion_3hr"
]

print("Categorical features:", len(categorical_features))
for feature in categorical_features:
    print(" -", feature)

print("\nNumerical features:", len(numerical_features))
for feature in numerical_features:
    print(" -", feature)

print("\nTotal features:",
      len(categorical_features) + len(numerical_features))

Categorical features: 3
 - Origin
 - Dest
 - Carrier

Numerical features: 14
 - Hour
 - DayOfWeek
 - Month
 - DayOfMonth
 - IsWeekend
 - Temperature
 - Feels_Like_Temperature
 - Altimeter_Pressure
 - Sea_Level_Pressure
 - Visibility
 - Wind_Speed
 - Wind_Gust
 - Precipitation
 - Ice_Accretion_3hr

Total features: 17


In [25]:
# ====================================================
# BUILD PREPROCESSING PIPELINE USING ONE HOT ENCODING
# ====================================================

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=True
            ),
            categorical_features
        ),
        (
            "numerical",
            StandardScaler(),
            numerical_features
        )
    ]
)

print("Preprocessing pipeline created successfully.")
print("\nCategorical processing:")
print("  Origin, Dest, Carrier → One-Hot Encoding")

print("\nNumerical processing:")
print("  14 numerical features → Standard Scaling")



Preprocessing pipeline created successfully.

Categorical processing:
  Origin, Dest, Carrier → One-Hot Encoding

Numerical processing:
  14 numerical features → Standard Scaling


In [26]:
# ============================================
# PREPARE TEMPORAL MODELING VIEWS
# ============================================

# Feature columns
X_columns = feature_columns

# Classification target
y_classification_column = "Cancelled"

# Regression target
y_regression_column = "DepDelayMinutes"

# Create row masks
train_mask = flightfore["Time"].dt.year == 2021
val_mask = flightfore["Time"].dt.year == 2022
test_mask = flightfore["Time"].dt.year == 2023

print("Temporal masks verified.")

print("\nTraining rows:", train_mask.sum())
print("Validation rows:", val_mask.sum())
print("Test rows:", test_mask.sum())

Temporal masks verified.

Training rows: 4230366
Validation rows: 5036769
Test rows: 5172164


In [27]:
# ============================================
# CHECK TARGETS BY TIME SPLIT
# ============================================

print("CLASSIFICATION TARGET")
print("=" * 50)

for name, mask in [
    ("Train (2021)", train_mask),
    ("Validation (2022)", val_mask),
    ("Test (2023)", test_mask)
]:
    print(f"\n{name}")
    print(flightfore.loc[mask, "Cancelled"].value_counts())
    print(
        "Cancellation rate:",
        f"{flightfore.loc[mask, 'Cancelled'].mean() * 100:.2f}%"
    )


print("\n\nREGRESSION TARGET")
print("=" * 50)

for name, mask in [
    ("Train (2021)", train_mask),
    ("Validation (2022)", val_mask),
    ("Test (2023)", test_mask)
]:
    regression_mask = (
        mask &
        (flightfore["Cancelled"] == False) &
        (flightfore["DepDelayMinutes"].notna())
    )

    print(f"\n{name}")
    print("Regression rows:", regression_mask.sum())
    print(
        "Mean delay:",
        f"{flightfore.loc[regression_mask, 'DepDelayMinutes'].mean():.2f} minutes"
    )
    print(
        "Median delay:",
        f"{flightfore.loc[regression_mask, 'DepDelayMinutes'].median():.2f} minutes"
    )

CLASSIFICATION TARGET

Train (2021)
Cancelled
False    4146550
True       83816
Name: count, dtype: int64
Cancellation rate: 1.98%

Validation (2022)
Cancelled
False    4875431
True      161338
Name: count, dtype: int64
Cancellation rate: 3.20%

Test (2023)
Cancelled
False    5086715
True       85449
Name: count, dtype: int64
Cancellation rate: 1.65%


REGRESSION TARGET

Train (2021)
Regression rows: 4146550
Mean delay: 13.50 minutes
Median delay: 0.00 minutes

Validation (2022)
Regression rows: 4875431
Mean delay: 17.15 minutes
Median delay: 0.00 minutes

Test (2023)
Regression rows: 5086715
Mean delay: 17.54 minutes
Median delay: 0.00 minutes


In [28]:
# ============================================
# CHECK CATEGORICAL CARDINALITY
# ============================================

print("Unique Origin airports:",
      flightfore.loc[train_mask, "Origin"].nunique())

print("Unique Destination airports:",
      flightfore.loc[train_mask, "Dest"].nunique())

print("Unique Carriers:",
      flightfore.loc[train_mask, "Carrier"].nunique())

print("\nCarrier names:")
print(
    flightfore.loc[train_mask, "Carrier"]
    .drop_duplicates()
    .sort_values()
    .to_list()
)

Unique Origin airports: 30
Unique Destination airports: 347
Unique Carriers: 17

Carrier names:
['Alaska Airlines Inc.', 'Allegiant Air', 'American Airlines Inc.', 'Delta Air Lines Inc.', 'Endeavor Air Inc.', 'Envoy Air', 'Frontier Airlines Inc.', 'Hawaiian Airlines Inc.', 'Horizon Air', 'JetBlue Airways', 'Mesa Airlines Inc.', 'PSA Airlines Inc.', 'Republic Airline', 'SkyWest Airlines Inc.', 'Southwest Airlines Co.', 'Spirit Air Lines', 'United Air Lines Inc.']
